# Transfer Learning for Image Classification (Flowers Dataset)

This notebook is my practice walkthrough of transfer learning: taking a CNN that Google already trained on ImageNet (1.4 million photos, 1000 categories) via TensorFlow Hub, and reusing it for a different, much smaller image classification problem, telling apart 5 flower species.

I'm doing this in two stages, on purpose, so I can actually see the difference transfer learning makes:

1. **Use the pretrained model exactly as-is**, with its original ImageNet classification head, and see what it predicts on both a generic photo and on my own flower photos (spoiler: it won't know what a tulip is, since that's not one of its 1000 trained categories).
2. **Swap out its classification head** for my own, and retrain just that small new piece on the flower dataset, keeping the pretrained convolutional layers frozen.

---

## 1. Setup

`tensorflow_hub` is the piece that isn't part of core TensorFlow. It's the library Google uses to publish ready-trained models (like MobileNetV2) as drop-in Keras layers, so I don't have to rebuild the architecture myself, just point at a URL and load it.

Everything else here is the usual stack: OpenCV for reading and resizing images, PIL for quick previews, and the standard Keras building blocks for the model I'll build later.

### A compatibility note before importing TensorFlow

Since TensorFlow 2.16, `pip install tensorflow` installs **Keras 3** by default, and Colab now ships with this version. `tensorflow_hub`'s `KerasLayer`, however, was built against the older Keras 2 interface, so wrapping a `hub.KerasLayer` inside a Keras 3 `Sequential` model raises `ValueError: Only instances of keras.Layer can be added to a Sequential model`, even though the code itself is correct.

The fix is to install `tf_keras` (the Keras 2 implementation, still maintained separately) and tell TensorFlow to route `tf.keras` through it for this session, via the `TF_USE_LEGACY_KERAS` environment variable. This has to be set **before** `tensorflow` is imported anywhere in the notebook, which is why it lives in its own cell, ahead of every other import.

In [ ]:
!pip install -q tensorflow_hub tf_keras

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [ ]:
import numpy as np
import cv2
import pathlib

import PIL.Image as Image
import matplotlib.pylab as plt

import tensorflow as tf
import tf_keras
import tensorflow_hub as hub

from tensorflow import keras
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)
print('Keras version (legacy, via tf_keras):', tf_keras.__version__)
print('TensorFlow Hub version:', hub.__version__)

> **Why `tf_keras.__version__` and not `keras.__version__`?** Once `TF_USE_LEGACY_KERAS=1` is set, `from tensorflow import keras` quietly points at `tf_keras` under the hood, but that compatibility shim doesn't expose a `__version__` attribute the same way the real `tf_keras` package does. Importing `tf_keras` directly, just for this version check, sidesteps that.

---

## 2. Using the Pretrained Model Exactly As-Is

Before I touch training at all, I want to see what this model can already do out of the box. `hub.KerasLayer(...)` pulls **MobileNetV2**, complete with its original 1000-class ImageNet classification head, straight from TensorFlow Hub and wraps it as a single Keras layer I can drop into a `Sequential` model.

`IMAGE_SHAPE = (224, 224)` matters because this isn't a suggestion, it's a requirement: MobileNetV2 was trained on 224x224 images, so every image I feed it has to be resized to match, or the shapes simply won't line up.

In [ ]:
IMAGE_SHAPE = (224, 224)

mobilenet_url = ('https://tfhub.dev/google/tf2-preview/mobilenet_v2/'
                  'classification/4')

classifier = tf.keras.Sequential([
    hub.KerasLayer(mobilenet_url, input_shape=IMAGE_SHAPE + (3,))
])

### Getting a test image

Rather than uploading a photo by hand every time I run this notebook, I'll pull a sample image straight from a public TensorFlow URL (the same trick used for the flowers dataset below), so this cell reproduces the same way for anyone re-running it, without needing to manually drop a file into Colab first.

In [ ]:
sample_image_url = ('https://storage.googleapis.com/download.tensorflow.org'
                     '/example_images/YellowLabradorLooking_new.jpg')
sample_image_path = tf.keras.utils.get_file('sample_dog.jpg', sample_image_url)

sample_image = Image.open(sample_image_path).resize(IMAGE_SHAPE)
sample_image

### Preparing the image for the model

Two things need to happen before this image can go into the classifier:

1. **Scale pixel values to `[0, 1]`.** The photo loads with pixel intensities from 0-255. Dividing by 255 puts them on the small, normalized scale the model expects.
2. **Add a batch dimension.** Keras models always expect a *batch* of images, shaped `(num_images, height, width, channels)`, even if I'm only predicting on one photo. `[np.newaxis, ...]` turns a single `(224, 224, 3)` image into a `(1, 224, 224, 3)` batch of one.

In [ ]:
sample_image = np.array(sample_image) / 255.0
print('Image shape:', sample_image.shape)

sample_batch = sample_image[np.newaxis, ...]
print('Batch shape:', sample_batch.shape)

### Running the prediction

The output shape tells the story here: `(1, 1001)` means one image in, and 1001 probability-like scores out, one per ImageNet class (the extra slot beyond the usual 1000 is a background class some versions of this model include). `np.argmax` picks out the index of the highest-scoring class, which is the model's actual prediction.

In [ ]:
prediction = classifier.predict(sample_batch)
print('Output shape:', prediction.shape)

predicted_index = np.argmax(prediction)
print('Predicted class index:', predicted_index)

### Turning that index into a readable label

The 1001 output positions correspond to a fixed, ordered list of ImageNet class names. That list isn't built into the model itself, it's a separate lookup file, so I download it once and index into it with the prediction above.

In [ ]:
labels_path = tf.keras.utils.get_file(
    'ImageNetLabels.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt'
)

with open(labels_path, 'r') as f:
    imagenet_labels = f.read().splitlines()

print('Predicted label:', imagenet_labels[predicted_index])

---

## 3. Loading the Flowers Dataset

Now for the dataset I actually care about. This is a small public dataset of flower photos, split into folders by species, hosted on the same storage bucket Google uses for its official TensorFlow tutorials. `get_file(..., untar=True)` downloads and extracts it in one step, and caches it locally so re-running this cell later won't re-download it.

In [ ]:
flowers_url = ('https://storage.googleapis.com/download.tensorflow.org/'
               'example_images/flower_photos.tgz')

data_dir = tf.keras.utils.get_file(
    'flower_photos', origin=flowers_url, cache_dir='.', untar=True
)
data_dir = pathlib.Path(data_dir)

print('Dataset extracted to:', data_dir)

In [ ]:
image_count = len(list(data_dir.glob('*/*.jpg')))
print('Total images in the dataset:', image_count)

### A quick look at the categories

Each species lives in its own subfolder, which is the standard layout for this kind of dataset, the folder name *is* the label. Before writing any loading code, I just want to eyeball a couple of photos to confirm the images look like what I expect.

In [ ]:
roses = list(data_dir.glob('roses/*'))
tulips = list(data_dir.glob('tulips/*'))

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(Image.open(str(roses[1])))
axes[0].set_title('rose')
axes[0].axis('off')
axes[1].imshow(Image.open(str(tulips[0])))
axes[1].set_title('tulip')
axes[1].axis('off')
plt.tight_layout()
plt.show()

---

## 4. Turning the Photos into Arrays

The model can't work with a folder of `.jpg` files directly, it needs numeric arrays, all the same size. I build two dictionaries first: one mapping each species name to the list of its image file paths, and one mapping each species name to the integer label it should train against.

In [ ]:
flowers_images_dict = {
    'roses': list(data_dir.glob('roses/*')),
    'daisy': list(data_dir.glob('daisy/*')),
    'dandelion': list(data_dir.glob('dandelion/*')),
    'sunflowers': list(data_dir.glob('sunflowers/*')),
    'tulips': list(data_dir.glob('tulips/*')),
}

flowers_labels_dict = {
    'roses': 0,
    'daisy': 1,
    'dandelion': 2,
    'sunflowers': 3,
    'tulips': 4,
}

for name, images in flowers_images_dict.items():
    print(f'{name:12s}: {len(images)} images')

### Reading and resizing every image

I loop through every category, read each photo with OpenCV (`cv2.imread`), and resize it to `224x224` to match what MobileNetV2 expects. This is the same resolution I used for the sample dog photo above, so both the pretrained classifier and my own retrained model further down will accept these arrays without any extra reshaping.

In [ ]:
X, y = [], []

for flower_name, image_paths in flowers_images_dict.items():
    for image_path in image_paths:
        img = cv2.imread(str(image_path))
        resized_img = cv2.resize(img, (224, 224))
        X.append(resized_img)
        y.append(flowers_labels_dict[flower_name])

X = np.array(X)
y = np.array(y)

print('X shape:', X.shape)
print('y shape:', y.shape)

---

## 5. Train/Test Split

I hold out a portion of the data purely for testing, so I can get an honest read on how the model performs on photos it never trained on, rather than judging it on data it's already seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

print('Training images:', X_train.shape[0])
print('Test images:', X_test.shape[0])

## 6. Scaling Pixel Values

Same reasoning as the sample dog photo earlier: raw pixel values sit on a 0-255 scale, and dividing by 255 brings them down to `[0, 1]`, which trains faster and more stably than leaving the raw values as they are.

In [ ]:
X_train_scaled = X_train / 255
X_test_scaled = X_test / 255

---

## 7. Testing the Untouched Pretrained Model on Flower Photos

Here's the part that motivates the rest of this notebook. The `classifier` I built in Section 2 still has its *original* ImageNet head attached, it has never seen my flower dataset. Let's see what it actually predicts when I feed it real photos from my dataset.

In [ ]:
sample_indices = [0, 1, 2]
resized_samples = np.array([
    cv2.resize(X[i], IMAGE_SHAPE) for i in sample_indices
])

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, i in zip(axes, sample_indices):
    ax.imshow(X[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
predictions = classifier.predict(resized_samples)
predicted_indices = np.argmax(predictions, axis=1)

for i, idx in zip(sample_indices, predicted_indices):
    print(f'Image {i} -> predicted as: {imagenet_labels[idx]}')

This is exactly the problem transfer learning is meant to solve. ImageNet's 1000 categories don't include "rose" or "tulip" as their own classes, so the model reaches for whatever ImageNet category looks visually closest, which is often wrong or oddly specific. The convolutional layers underneath still know how to *see* (edges, textures, colors), it's only the final decision layer that's mismatched to my task. That's the piece I'm about to replace.

---

## 8. Transfer Learning: Swapping the Classification Head

This time I load a *different* version of MobileNetV2 from TensorFlow Hub: the **feature vector** model instead of the **classification** model. The difference matters:

- The classification version (Section 2) ends in a 1001-way softmax over ImageNet categories.
- The feature vector version stops one step earlier, and just outputs a description vector for whatever it sees, with no final decision layer at all.

`trainable=False` freezes every weight in this feature extractor. I'm deliberately keeping the part that already knows how to recognize shapes and textures locked in place, and only adding a small new piece that I'll actually train.

In [ ]:
feature_extractor_url = ('https://tfhub.dev/google/tf2-preview/mobilenet_v2/'
                          'feature_vector/4')

pretrained_feature_extractor = hub.KerasLayer(
    feature_extractor_url, input_shape=(224, 224, 3), trainable=False
)

### Attaching my own classification head

This is the entire trainable part of the model: one `Dense` layer with 5 outputs, one per flower species. It sits directly on top of the frozen feature extractor, so the model overall goes: image -> frozen MobileNetV2 features -> my own 5-way decision layer.

The `.summary()` output is worth checking here specifically to confirm how few parameters are actually trainable, since the vast majority of the network's weights belong to the frozen feature extractor.

In [ ]:
num_of_flowers = 5

transfer_model = tf.keras.Sequential([
    pretrained_feature_extractor,
    tf.keras.layers.Dense(num_of_flowers)
])

transfer_model.summary()

### Compiling and training

The final `Dense` layer has no activation function, so it outputs raw scores rather than probabilities. `from_logits=True` tells the loss function to apply softmax internally, which is a more numerically stable way to handle this than adding a separate `softmax` activation myself.

Since only that one small `Dense` layer has trainable weights, this trains fast, just 5 epochs gets a strong result here, compared to what it would take to train a full CNN from scratch on this same amount of data.

In [ ]:
transfer_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['acc'],
)

transfer_model.fit(X_train_scaled, y_train, epochs=5)

### Evaluating on the held-out test set

This is the real measure of whether the retrained head actually generalized, using the test images set aside back in Section 5, which the model has never trained on.

In [ ]:
test_loss, test_accuracy = transfer_model.evaluate(X_test_scaled, y_test)
print(f'Test accuracy: {test_accuracy:.4f}')

---

## 9. What This Actually Demonstrates

- **Section 2 and 7** used MobileNetV2 exactly as Google trained it, and it predicted confidently but incorrectly on flower photos, since those categories were never part of its original 1000 classes.
- **Section 8** reused the same pretrained convolutional layers (frozen, unchanged) but replaced the final decision layer with one trained specifically on my 5 flower categories.
- The result: a handful of epochs and one trainable `Dense` layer got a solid accuracy on flowers, without ever training a CNN from scratch, because the hard part (learning to see general visual patterns) was already solved by the pretrained model.